# 🧍 원하는 포즈로 이미지 만드는 도구

참조 사진 속 **사람의 자세(관절 위치)** 를 뽑아서, 그 자세 그대로 **다른 인물·장면** 을 만드는 튜토리얼입니다.

## 전체 흐름 (4단계)
1. **포즈 추출**: OpenPose가 사진에서 관절과 뼈대를 뽑습니다. (막대 인형 그림이 나와요)
2. **조건 넣기**: ControlNet이 그 막대 인형 그림을 "이 자세로 그려"라는 조건으로 바꿉니다.
3. **이미지 생성**: 확산 모델이 프롬프트의 내용 + 막대 인형의 자세로 그림을 만듭니다.
4. **저장·비교**: 결과를 `samples/`에 저장하고, 조건을 바꿔 가며 비교합니다.

## 왜 이 모델을 골랐나
- 교안의 출발점은 FLUX.2-klein 4B 노트입니다. 그런데 교안에 적힌 대로 FLUX.2에는 BFL 공식 ControlNet이 없고, 커뮤니티 ControlNet은 FLUX.2-dev 기반이라 무료 Colab T4에는 무겁습니다.
- 그래서 과제에서 허용한 "비슷한 모델"로 **Stable Diffusion 1.5 + ControlNet v1.1 OpenPose** 를 씁니다. 포즈 ControlNet이 가장 널리 쓰이고, 무료 T4에서도 한 장에 몇 초면 됩니다.
- 라이선스: CreativeML OpenRAIL-M (과제·학습용으로 사용).

## 실행 전 준비
- 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르세요.
- 위에서부터 셀을 **순서대로** 실행하세요. (셀 왼쪽 ▶ 버튼 또는 Shift+Enter)

## 1. GPU 확인
그래픽카드가 붙었는지, 메모리(VRAM)가 얼마인지 확인합니다. GPU가 없으면 여기서 멈춥니다.

In [ ]:
# [이 셀이 하는 일] GPU가 붙었는지 확인하고 이름과 메모리(VRAM)를 보여 줍니다.
import torch

if not torch.cuda.is_available():
    raise SystemExit("GPU가 안 붙었습니다. 런타임 → 런타임 유형 변경 → T4 GPU로 바꾸고 다시 실행하세요.")

p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} / VRAM {p.total_memory / 1024**3:.1f}GB")

## 2. 도구 설치
- `diffusers`: 확산 모델을 이름만으로 불러 실행해 주는 도구
- `controlnet_aux`: 사진에서 포즈(관절)를 뽑아 주는 도구

1~2분 걸릴 수 있습니다.

In [ ]:
# [이 셀이 하는 일] 이미지 생성(diffusers)과 포즈 추출(controlnet_aux)에 필요한 도구를 설치합니다.
!pip install -q -U diffusers transformers accelerate controlnet_aux
print("설치 완료")

## 3. 모델 불러오기
세 가지를 불러옵니다.
- **OpenPose 검출기**: 사진 → 막대 인형(관절) 그림
- **ControlNet (OpenPose용)**: 막대 인형 그림 → "이 자세로 그려"라는 조건
- **Stable Diffusion 1.5**: 조건 + 프롬프트 → 그림

처음 한 번은 내려받느라 몇 분 걸립니다.

In [ ]:
# [이 셀이 하는 일] 포즈 검출기, 포즈용 ControlNet, 이미지 생성 모델을 내려받아 GPU에 올립니다.
import torch
from controlnet_aux import OpenposeDetector
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

# 1) 사진에서 관절을 뽑는 검출기
openpose = OpenposeDetector.from_pretrained("lllyasviel/Annotators")

# 2) 관절 그림을 조건으로 쓰는 ControlNet (SD 1.5용 OpenPose)
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_openpose", torch_dtype=torch.float16
)

# 3) 그림을 만드는 Stable Diffusion 1.5
#    원래 저장소(runwayml)가 내려가 있으면 공식 미러 저장소로 다시 시도합니다.
BASE_MODEL = None
for repo in ["stable-diffusion-v1-5/stable-diffusion-v1-5", "runwayml/stable-diffusion-v1-5"]:
    try:
        pipe = StableDiffusionControlNetPipeline.from_pretrained(
            repo, controlnet=controlnet, torch_dtype=torch.float16
        )
        BASE_MODEL = repo
        break
    except Exception as e:
        print(f"{repo} 불러오기 실패, 다음 저장소로 시도합니다: {e}")

if BASE_MODEL is None:
    raise SystemExit("기본 모델을 불러오지 못했습니다. 인터넷 연결이나 모델 저장소 이름을 확인하세요.")

# 빠르게 그리는 스케줄러로 바꾸고 GPU에 올립니다.
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
print("모델 준비 완료:", BASE_MODEL)

## 4. 결과 저장 도우미 함수
이후 셀에서 반복해서 쓸 작은 함수들입니다. 사진 크기 맞추기, 포즈 뽑기, 그림 만들기, 나란히 보여 주기를 담당합니다.

In [ ]:
# [이 셀이 하는 일] 사진 크기 맞추기 / 포즈 추출 / 이미지 생성 / 나란히 보기 함수를 만듭니다.
import os, json
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 그래프 제목의 한글이 네모로 깨지지 않게 나눔고딕 글꼴을 설치·등록합니다.
os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
try:
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"
except Exception as e:
    print("한글 글꼴 설정 실패 (결과 이미지에는 영향 없음):", e)

os.makedirs("samples", exist_ok=True)
LOG = []   # 어떤 조건으로 만들었는지 기록 (재현용)

def fit_size(img, max_side=640):
    """사진을 똑바로 세우고, 긴 변을 max_side 이하로 줄이고, 가로·세로를 8의 배수로 맞춥니다. (모델이 8의 배수를 좋아해요)"""
    img = ImageOps.exif_transpose(img).convert("RGB")  # 휴대폰 사진의 회전 정보(EXIF)를 반영해 똑바로 세웁니다.
    w, h = img.size
    s = min(1.0, max_side / max(w, h))
    w, h = int(w * s) // 8 * 8, int(h * s) // 8 * 8
    return img.resize((w, h))

def get_pose(img):
    """사진에서 관절(막대 인형) 그림을 뽑습니다. 손과 얼굴 점까지 뽑습니다."""
    pose = openpose(img, hand_and_face=True)
    return pose.resize(img.size)

def generate(pose_img, prompt, seed=42, steps=25, guidance=7.5, pose_strength=1.0, negative_prompt=""):
    """포즈 그림 + 프롬프트로 이미지를 한 장 만듭니다.
    - seed: 출발 잡음 번호. 같게 두면 프롬프트·포즈 효과만 비교할 수 있어요.
    - steps: 잡음을 걷어내는 횟수 (이 모델은 증류 모델이 아니라 20~30이 적당)
    - guidance: 프롬프트를 얼마나 꽉 따를지 (이 모델은 7~8이 보통)
    - pose_strength: 포즈를 얼마나 꽉 따를지 (0~2, 기본 1.0)
    """
    g = torch.Generator("cuda").manual_seed(seed)
    out = pipe(
        prompt, image=pose_img, num_inference_steps=steps, guidance_scale=guidance,
        controlnet_conditioning_scale=pose_strength, negative_prompt=negative_prompt or None,
        generator=g,
    ).images[0]
    return out

def record(name, prompt, seed, **kw):
    LOG.append({"file": name, "prompt": prompt, "seed": seed, "model": BASE_MODEL,
                "controlnet": "lllyasviel/control_v11p_sd15_openpose", **kw})

def show(images, titles):
    """여러 장을 가로로 나란히 보여 줍니다."""
    plt.figure(figsize=(5 * len(images), 5))
    for i, (im, t) in enumerate(zip(images, titles)):
        plt.subplot(1, len(images), i + 1); plt.imshow(im); plt.title(t, fontsize=9); plt.axis("off")
    plt.show()

print("함수 준비 완료")

## 5. 참조 사진 1 올리기
**사람의 온몸이나 상반신이 잘 보이는 사진** 을 고르세요.
- 직접 찍은 사진이나 자유 이용 사진(예: Unsplash, Pexels)을 권장합니다.
- 다른 사람 얼굴 사진을 쓸 때는 허락을 받았는지 확인하세요. 결과물엔 자세만 쓰입니다.
- 손이 작게 찍힌 사진은 손 관절이 흔들릴 수 있어요.
- **사람이 화면 절반 이상을 차지하고 밝게 나온 사진**이 잘 됩니다. 사람이 작거나 역광이면 관절을 못 찾아 막대 인형이 까맣게 나와요.
- 카카오톡으로 받은 사진은 회전 정보가 없어 옆으로 누울 수 있어요. 휴대폰 원본을 쓰세요.

셀을 실행하면 **파일 선택** 버튼이 나옵니다.

In [ ]:
# [이 셀이 하는 일] 내 컴퓨터에서 참조 사진을 올리고, 크기를 맞춘 뒤 samples/ref_01.png로 저장합니다.
from google.colab import files
up = files.upload()
ref1 = fit_size(Image.open(list(up.keys())[0]))
ref1.save("samples/ref_01.png")
print("참조 사진 크기:", ref1.size)

## 6. 포즈 뽑기
사진에서 관절을 뽑아 **막대 인형 그림** 으로 만들고 `samples/pose_01.png`로 저장합니다.
관절이 엉뚱하게 잡히면 다른 사진으로 바꾸는 게 가장 빠릅니다.

In [ ]:
# [이 셀이 하는 일] 참조 사진에서 관절(막대 인형)을 뽑아 samples/pose_01.png로 저장하고 나란히 보여 줍니다.
pose1 = get_pose(ref1)
pose1.save("samples/pose_01.png")
show([ref1, pose1], ["참조 사진", "뽑아낸 포즈"])

## 7. 첫 결과 만들기
포즈 1 + 프롬프트 하나로 첫 장을 만들어 `samples/output_01.png`로 저장합니다.

> 💡 **프롬프트 팁**: 교안의 여섯 칸(피사체·동작·카메라·조명·환경·스타일)을 채우세요. 단, 이 모델의 텍스트 인코더(CLIP)는 **약 77토큰(대략 50~60단어)** 까지만 읽고 나머지는 잘라 버립니다. 그래서 교안의 60~120단어보다 **짧고 핵심만** 쓰는 게 좋아요. 자세는 포즈가 정하니, 프롬프트엔 **누가, 어디서, 어떤 빛과 스타일로** 를 적으면 됩니다.

In [ ]:
# [이 셀이 하는 일] 포즈 1과 프롬프트로 첫 이미지를 만들어 samples/output_01.png로 저장합니다.
SEED = 42
PROMPT_1 = ("A factory worker in a white safety helmet and blue overalls, inside a bright manufacturing plant, "
            "soft morning light from tall windows on the left, photorealistic documentary photography, muted colors")

out1 = generate(pose1, PROMPT_1, seed=SEED)
out1.save("samples/output_01.png")
record("output_01.png", PROMPT_1, SEED, pose="pose_01.png")
show([ref1, pose1, out1], ["참조 사진", "포즈", "결과"])

## 8. 실험 A: 같은 포즈 + 다른 프롬프트
**포즈와 시드는 그대로**, 프롬프트만 바꿉니다. 자세는 유지되는지, 무엇이 바뀌는지 봅니다.
프롬프트는 마음대로 바꿔도 됩니다.

In [ ]:
# [이 셀이 하는 일] 포즈 1과 시드를 고정하고, 프롬프트만 바꿔 3장을 만들어 저장합니다.
PROMPTS_A = [
    "An astronaut in a white space suit on the surface of the moon, earth in the black sky, cinematic lighting, photorealistic",
    "A medieval knight in silver armor in a castle courtyard, overcast daylight, oil painting style",
    "A cheerful chef in a white uniform in a busy restaurant kitchen, warm tungsten light, documentary photo",
]
outs_a = []
for i, pr in enumerate(PROMPTS_A, start=1):
    im = generate(pose1, pr, seed=SEED)
    name = f"exp_a_{i:02d}.png"
    im.save(f"samples/{name}")
    record(name, pr, SEED, pose="pose_01.png", experiment="A: 같은 포즈, 다른 프롬프트")
    outs_a.append(im)
show([pose1] + outs_a, ["포즈 1 (고정)"] + [f"프롬프트 {i}" for i in range(1, 4)])

## 9. 참조 사진 2 올리기
실험 B를 위해 **자세가 확실히 다른 사진** 을 하나 더 올립니다. (예: 첫 사진이 서 있으면, 이번엔 앉거나 팔을 든 자세)

In [ ]:
# [이 셀이 하는 일] 두 번째 참조 사진을 올리고 포즈를 뽑아 samples/pose_02.png로 저장합니다.
up2 = files.upload()
ref2 = fit_size(Image.open(list(up2.keys())[0]))
ref2.save("samples/ref_02.png")
pose2 = get_pose(ref2)
pose2.save("samples/pose_02.png")
show([ref2, pose2], ["참조 사진 2", "뽑아낸 포즈 2"])

## 10. 실험 B: 다른 포즈 + 같은 프롬프트
**프롬프트와 시드는 그대로**, 포즈만 바꿉니다. 포즈만 바꿨을 때 결과가 얼마나 달라지는지 봅니다.

In [ ]:
# [이 셀이 하는 일] 프롬프트 1과 시드를 고정하고, 포즈 2로 이미지를 만들어 samples/output_02.png로 저장합니다.
out2 = generate(pose2, PROMPT_1, seed=SEED)
out2.save("samples/output_02.png")
record("output_02.png", PROMPT_1, SEED, pose="pose_02.png", experiment="B: 다른 포즈, 같은 프롬프트")
show([pose1, out1, pose2, out2], ["포즈 1", "결과 (포즈 1)", "포즈 2", "결과 (포즈 2)"])

## 11. (선택) 실험 C: 포즈를 얼마나 꽉 따를지
`pose_strength` 값을 바꿔 봅니다. 낮으면 프롬프트 쪽이, 높으면 포즈 쪽이 세집니다.

In [ ]:
# [이 셀이 하는 일] 같은 포즈·프롬프트·시드에서 포즈 세기만 0.5 / 1.0 / 1.5로 바꿔 비교합니다.
outs_c = []
for s in [0.5, 1.0, 1.5]:
    im = generate(pose1, PROMPT_1, seed=SEED, pose_strength=s)
    name = f"exp_c_strength_{s}.png"
    im.save(f"samples/{name}")
    record(name, PROMPT_1, SEED, pose="pose_01.png", pose_strength=s, experiment="C: 포즈 세기")
    outs_c.append(im)
show(outs_c, ["포즈 세기 0.5", "포즈 세기 1.0", "포즈 세기 1.5"])

## 12. 기록 저장하고 내려받기
어떤 프롬프트·시드·모델로 만들었는지 `samples/log.json`에 저장하고, `samples/` 폴더를 압축해 내 컴퓨터로 내려받습니다.
내려받은 파일을 GitHub 저장소의 `samples/` 폴더에 올리면 됩니다.

In [ ]:
# [이 셀이 하는 일] 생성 기록을 log.json으로 저장하고, samples 폴더를 zip으로 묶어 내려받습니다.
with open("samples/log.json", "w", encoding="utf-8") as f:
    json.dump(LOG, f, ensure_ascii=False, indent=2)
!zip -q -r samples.zip samples
files.download("samples.zip")
print("내려받기 시작. 저장된 이미지 수:", len([n for n in os.listdir('samples') if n.endswith('.png')]))

## 13. 실험 관찰 기록 ✍️
> 시드 42, 스텝 25, 가이던스 7.5로 고정하고 돌림. T4에서 한 장에 6~8초 걸렸음.

### 실험 A: 같은 포즈, 다른 프롬프트
- 바꾼 거: 프롬프트만 (우주인 / 기사 / 요리사). 포즈랑 시드는 그대로
- 자세: 셋 다 똑바로 서서 팔 내린 자세 그대로 나옴. 옷이랑 배경이 완전 바뀌어도 자세는 안 바뀌더라
- 크게 바뀐 거: 옷, 배경, 분위기 전부
- 잘 안 따라온 거
  - 우주인: 달 표면은 나왔는데 "까만 하늘에 지구"는 없음. 하늘도 그냥 회색
  - 기사: 성문 앞은 맞는데 "유화 스타일"이 아니라 그냥 사진처럼 나옴
  - 요리사: 웃는 얼굴이랑 바쁜 주방은 맞는데 "따뜻한 주황빛 조명"이 아니라 하얀 빛
  - 사람, 옷, 장소 같은 큰 건 잘 따라오는데 하늘 속 작은 물체, 화풍, 조명 색 같은 디테일은 잘 빠짐

### 실험 B: 다른 포즈, 같은 프롬프트
- 바꾼 거: 포즈만 (서 있는 자세 → 바닥에 앉아서 폰 들고 있는 자세). 프롬프트랑 시드는 그대로
- 자세: 앉은 자세, 고개 기울인 거, 한 손은 얼굴 옆, 한 손은 가슴 앞까지 잘 따라감
- 신기했던 거
  - 막대 인형이 화면을 꽉 채우니까 결과도 가까이서 찍은 사진처럼 나옴. 포즈가 자세뿐 아니라 구도랑 인물 크기까지 정함
  - 프롬프트에 폰 없는데 손에 폰이 생김. 손가락 모양이 뭔가 쥔 손이라 모델이 알아서 채운 듯
  - 성별을 안 적었더니 결과 1은 남자, 결과 2는 여자로 나옴. 안 적은 칸은 모델이 정한다는 거 그대로
- 잘 안 따라온 거: 색이 엉뚱한 데 붙음. 결과 1은 "파란 작업복"이라고 했는데 작업복은 하얗고 셔츠가 파랬고, 결과 2는 작업복은 파랗게 맞았는데 "하얀 안전모"가 파랗게 나옴. 창문도 왼쪽이라고 했는데 결과 1은 오른쪽에 있었음

### 실험 C: 포즈 세기 (0.5 / 1.0 / 1.5)
- 0.5: 자세는 거의 같음. 사람이 좀 더 크고 통통하고, 천장 조명이 노랗게 바뀌는 등 배경이 더 자유로움
- 1.0: 처음 결과(output_01)랑 똑같음. 설정이 다 같으니까
- 1.5: 사람이 더 마르고 얼굴이 좀 어색해짐. 작업복에 이상한 하늘색 띠도 생김
- 서 있는 자세는 원래 흔한 자세라 약하게 걸어도 비슷하게 나온 듯. 너무 세게 걸면 오히려 그림이 어색해짐. 조건 따르는 힘이랑 자연스러움 사이 균형이라는 게 이거구나 싶었음

### 정리: 어느 쪽을 바꿨을 때 결과가 더 크게 달라졌나?
- 포즈 바꿨을 때가 훨씬 크게 달라짐. 자세뿐 아니라 구도, 인물 크기, 손에 든 물건까지 바뀜
- 프롬프트 바꾸면 옷, 배경, 분위기는 바뀌는데 자세는 그대로 유지됨

### 한계랑 다음에 해볼 거
- 처음 올린 사진은 사람이 작고 역광이라 그림자처럼 찍혀서 포즈를 하나도 못 뽑음 (막대 인형이 까맣게 나옴). 카톡으로 받은 사진이라 회전 정보도 없어서 옆으로 누워서 나왔음. 사람이 크고 밝게 나온 사진으로 바꾸니까 바로 됨
- 폰이 얼굴이랑 손을 가리니까 얼굴 점이 엉클어지고, 사진 아래가 잘리니까 다리가 한쪽만 잡힘
- 이 모델은 프롬프트를 77토큰까지만 읽어서 짧게 써야 함. 그래서 색이나 조명 같은 디테일이 더 잘 빠지는 듯
- 다음엔 색 지시를 옷마다 더 분명하게 나눠 써보고, 시드 여러 개로 뽑아보고, 팔 든 자세처럼 덜 흔한 자세로 포즈 세기 다시 비교해보고 싶음
